Author: zhanghong, liuhui
Last updated: 2026-11-14

# 案例：画像建模与偏好记忆

## 核心学习目标

本案例以用户画像建模与偏好记忆为核心目标。通过结构化抽取、相似记忆合并、向量存储和混合检索，将非结构化对话转换为可复用的画像事实与偏好记忆，并在后续交互中与短期上下文共同注入提示词，使系统能够持续积累用户特征并据此提供个性化响应。

## 案例设计

案例按照“记忆抽取—记忆合并—记忆存储—混合检索—上下文使用”的流程组织，每个阶段由独立 Python 模块实现，并逐步构建可持续更新的用户画像和偏好记忆。Notebook 随后依次展示各阶段的调用方式、模块边界和执行结果。

**学习重点**：
1. 理解如何从连续交互中抽取稳定的用户事实、偏好和约束，并形成可持续更新的用户画像。
2. 掌握长期记忆与短期上下文在职责和生命周期上的差异。
3. 理解向量相似度与 BM25 混合检索如何召回画像事实和偏好记忆。
4. 掌握将检索结果注入提示词并据此生成个性化回答的完整流程。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 记忆抽取 | 生成摘要、关键点和重要性评分 | `MemoryExtractor.extract` |
| 记忆合并 | 合并内容相近的记忆并保留关键信息 | `MemoryMerger.merge` |
| 记忆存储 | 完成去重、合并和 Chroma 索引更新 | `MemoryStore.ingest_long_term` |
| 混合检索 | 融合向量检索、BM25 与重排分数 | `MemoryRetriever.search` |
| 上下文使用 | 组合短期对话与长期记忆并构建提示词 | `MemoryContextBuilder.build_prompt` |


## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 Notebook 前，请在仓库根目录执行 `uv pip install -r Chapter-3/requirements.txt`，并将 Notebook 内核设置为该 `.venv` 对应的 Python 解释器。环境变量统一从仓库根目录的 `.env` 加载。

In [1]:
import json
import sys
from datetime import datetime
from pathlib import Path

chapter3_candidate = Path.cwd()
if not (chapter3_candidate / "llm.py").exists():
    chapter3_candidate = Path.cwd() / "Chapter-3"

if (chapter3_candidate / "llm.py").exists():
    sys.path.insert(0, str(chapter3_candidate))

from llm import CHAPTER3, PROJECT_ROOT, load_project_env

env_path = load_project_env()
ROOT = PROJECT_ROOT

print("Chapter-3:", CHAPTER3)
print("项目根目录:", ROOT)
print("环境文件:", env_path)
print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

Chapter-3: E:\Book\agent-systems-book\Chapter-3
项目根目录: E:\Book\agent-systems-book
环境文件: E:\Book\agent-systems-book\.env
当前时间: 2026-11-14 21:03:53 +0800


## 1. 系统架构

系统包含记忆写入和记忆读取两条路径：

```
用户输入 ──► 记忆抽取 ──► 去重与合并 ──► 向量存储
用户查询 ──► 向量检索与 BM25 ──► 结果重排 ──► 上下文注入 ──► 模型生成
```

写入路径将对话内容转换为结构化记忆并持久化；读取路径根据当前问题召回相关记忆，并与短期对话共同构成模型上下文。


## 2. 模块初始化

本案例按照职责拆分为独立模块：`memory_extract.py`、`memory_merge.py`、`memory_store.py`、`memory_retrieve.py` 和 `memory_use.py`。各模块通过明确的输入输出衔接，避免把抽取、存储和检索逻辑集中在一个类中。


In [2]:
# Import each stage module in memory lifecycle order
from llm import create_llm, extract_response_text
from memory_extract import MemoryExtractor
from memory_merge import MemoryMerger
from memory_store import get_or_reset_store
from memory_retrieve import MemoryRetriever
from memory_use import MemoryContextBuilder

print("已加载：抽取、合并、存储、检索、使用和工作流模块")


已加载：抽取、合并、存储、检索、使用和工作流模块


## 3. 初始化模型与基础对象

首先从项目根目录环境创建对话模型，并初始化记忆抽取器和合并器。存储、检索与上下文模块只依赖这些对象，不直接读取模型配置。


In [3]:
llm = create_llm()
extractor = MemoryExtractor(llm)
merger = MemoryMerger(llm)

print("模型、记忆抽取器和记忆合并器已初始化")


模型、记忆抽取器和记忆合并器已初始化


## 4. 模块 1：记忆抽取

本模块调用 `MemoryExtractor.extract`，通过大模型将非结构化文本转换为包含摘要、关键点和重要性评分的结构化记忆。模型请求显式设置 `response_format={"type": "json_object"}`，由服务端约束 JSON 输出，而不是依赖提示词反复重试。运行结果会标注 `extraction_method=llm`；如果模型调用或 JSON 解析失败，异常会直接向上抛出。


In [4]:
user_input = "我喜欢住海景房，预算每晚不超过800元，尽量距离景区近"
extracted = extractor.extract(user_input)

print("【原文】", user_input)
print("【抽取方式】 大模型抽取 (llm)")
print("【抽取结果】")
print(json.dumps({k: extracted.get(k) for k in ("summary", "key_points", "importance", "extraction_method")}, ensure_ascii=False, indent=2))

【原文】 我喜欢住海景房，预算每晚不超过800元，尽量距离景区近
【抽取方式】 大模型抽取 (llm)
【抽取结果】
{
  "summary": "用户偏好预算800元以内、靠近景区的海景房。",
  "key_points": [
    "喜欢住海景房",
    "预算每晚不超过800元",
    "尽量距离景区近"
  ],
  "importance": 0.9,
  "extraction_method": "llm"
}


## 5. 模块 2：记忆合并

本模块调用 `MemoryMerger.merge`，将语义或内容相近的多条记忆整合为一条综合记忆，同时保留各条记忆中的重要信息，以减少重复存储。


In [5]:
old_memory = {
    "id": "mem-old-001",
    "content": "用户喜欢住海景房，预算每晚不超过800元",
    "summary": "用户偏好海景住宿且预算有限",
    "key_points": ["喜欢海景房", "预算800元以内", "每晚"],
    "importance": 0.8,
}
new_memory = extractor.extract("用户说以后订酒店都要海景房，价格别超过800")
merged = merger.merge([old_memory, new_memory])

print("合并后:", json.dumps({k: merged[k] for k in ("summary", "key_points", "importance")}, ensure_ascii=False, indent=2))

合并后: {
  "summary": "用户偏好预订海景房，且要求每晚价格不超过800元。",
  "key_points": [
    "偏好房型：海景房",
    "预算限制：每晚不超过800元"
  ],
  "importance": 0.5
}


## 6. 模块 3：记忆存储

本模块通过 `MemoryStore.ingest_long_term` 将记忆写入 Chroma。写入过程包括结构化抽取、相似记忆检测、按需合并以及检索索引更新。

> **运行说明**：`get_or_reset_store` 默认复用当前 Notebook 会话中的存储实例，并清空集合中的已有数据。重启内核后，如确需重建持久化目录，可将 `FORCE_REBUILD_DIRECTORY` 设置为 `True`。


In [6]:
USER_ID = "lact_lucky"
CHROMA_DIR = ROOT / "chroma_notebook_dem"
FORCE_REBUILD_DIRECTORY = False  # Set to True after Restart Kernel to rebuild the directory

store = get_or_reset_store(
    user_id=USER_ID,
    persist_directory=CHROMA_DIR,
    extractor=extractor,
    merger=merger,
    embedding_model="dashscope",
    collection_name="notebook_long_term_v1",
    force_rebuild_directory=FORCE_REBUILD_DIRECTORY,
)

samples = [
    ("用户喜欢住海景房，预算每晚不超过800元", "preference", 0.8),
    ("用户计划下个月去三亚旅游", "fact", 0.5),
    ("用户偏好海景住宿，价格控制在800以内", "preference", None),  # Near-duplicate; merge it
]
for text, mtype, imp in samples:
    mid = store.ingest_long_term(text, memory_type=mtype, importance=imp)
    print(f"写入 id={mid[:8]}… | {mtype} | {text[:20]}…")
print(f"\n库内条数: {len(store.documents)}（期望 2，第3条应合并）")

写入 id=8943ee2f… | preference | 用户喜欢住海景房，预算每晚不超过800元…
写入 id=e2544cb8… | fact | 用户计划下个月去三亚旅游…
写入 id=8943ee2f… | preference | 用户偏好海景住宿，价格控制在800以内…

库内条数: 2（期望 2，第3条应合并）


In [7]:
print("=== 库内记忆 ===")
for i, (doc_id, doc, meta) in enumerate(zip(store.ids, store.documents, store.metadatas), 1):
    print(f"[{i}] {meta.get('summary')} | type={meta.get('memory_type')} | imp={meta.get('importance')}")

=== 库内记忆 ===
[1] 用户偏好海景住宿，且预算上限为每晚800元。 | type=preference | imp=0.5
[2] 用户计划下个月前往三亚旅游。 | type=fact | imp=0.5


In [8]:
def show_user_profile(user_id: str) -> None:
    """Print the user profile grouped by preference, fact, and other dimensions."""
    records = store.get_user_records(user_id)
    groups = {"preference": "偏好", "fact": "事实", "general": "其他"}
    profile = {label: [] for label in groups.values()}

    for record in records:
        memory_type = record.get("memory_type", "general")
        label = groups.get(memory_type, "其他")
        summary = record.get("summary") or record.get("content", "")
        importance = float(record.get("importance", 0.5))
        profile[label].append(f"{summary}（重要性：{importance:.2f}）")

    print("=" * 16, "用户画像", "=" * 16)
    print("用户 ID:", user_id)
    print("画像记忆数:", len(records))
    for label, items in profile.items():
        if not items:
            continue
        print(f"\n【{label}】")
        for item in items:
            print(f"- {item}")


show_user_profile(USER_ID)

================ 用户画像 ================
用户 ID: lact_lucky
画像记忆数: 2

【偏好】
- 用户偏好海景住宿，且预算上限为每晚800元。（重要性：0.50）

【事实】
- 用户计划下个月前往三亚旅游。（重要性：0.50）


## 7. 模块 4：混合检索

检索结果按照以下步骤计算：

1. **vector_sim**：将 Chroma 距离转换为相似度，用于衡量查询与记忆的语义接近程度。
2. **rrf_score**：融合向量检索排名与 jieba BM25 排名，仅使用名次信息。
3. **base score**：使用 `rrf_score × vector_sim` 将排名结果与语义相似度结合。
4. **final_score**：使用 `base × time_decay × (0.5 + importance)` 引入时间衰减和重要性权重。

本模块通过 `MemoryRetriever` 执行混合检索，并以酒店偏好查询为例展示向量召回、混合排序及各分项分数的计算过程。


In [9]:
retriever = MemoryRetriever(
    store,
    top_k=5,
    similarity_threshold=0.0,
)


def show_search_results(query: str):
    print("=" * 72)
    print(f"QUERY: {query}")
    dense = retriever.dense_search(query, USER_ID, top_k=5)
    print("\n【向量检索】")
    for item in dense:
        meta = item["metadata"]
        print(f"  {meta.get('summary','')[:36]} | dist={item['distance']:.4f} | vector_sim={item['vector_sim']:.4f}")

    hits = retriever.search(query, user_id=USER_ID, top_k=5)
    print("\n【混合检索 MemoryRetriever.search — 分项 score】")
    for h in hits:
        print(
            f"  [{h['memory_type']}] {h['summary']}\n"
            f"    vector_sim={h['vector_sim']:.4f}  "
            f"rrf_score={h['rrf_score']:.4f}  "
            f"final_score={h['final_score']:.4f}"
        )
    return hits


hits_hotel = show_search_results("你还记得我喜欢住什么样的酒店吗？")
print()
QUERY = "你还记得我喜欢住什么样的酒店吗？"
hits = hits_hotel  # Module 5 uses the hotel-query results by default

QUERY: 你还记得我喜欢住什么样的酒店吗？


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\liuhui36\AppData\Local\Temp\jieba.cache



【向量检索】
  用户偏好海景住宿，且预算上限为每晚800元。 | dist=0.4564 | vector_sim=0.7718
  用户计划下个月前往三亚旅游。 | dist=0.4919 | vector_sim=0.7540


Loading model cost 0.357 seconds.
Prefix dict has been built successfully.



【混合检索 MemoryRetriever.search — 分项 score】
  [preference] 用户偏好海景住宿，且预算上限为每晚800元。
    vector_sim=0.7718  rrf_score=0.0167  final_score=0.0129
  [fact] 用户计划下个月前往三亚旅游。
    vector_sim=0.7540  rrf_score=0.0164  final_score=0.0124



## 8. 模块 5：上下文使用

本模块通过 `MemoryContextBuilder` 将当前线程的短期对话与检索得到的长期记忆组合为提示词，再直接调用 `llm.invoke` 生成回答。


In [10]:
context_builder = MemoryContextBuilder()
THREAD_ID = "thread_notebook_001"
context_builder.record_turn(THREAD_ID, "你好，我想规划一次旅行", "你好！请问您想去哪里旅行呢？")
prompt = context_builder.build_prompt(THREAD_ID, QUERY, hits=hits)
print(prompt)

你是一个具备长期记忆的智能助手。请结合【最近对话】与【相关长期记忆】回答用户问题。

【最近对话】（按时间顺序）
user: 你好，我想规划一次旅行
assistant: 你好！请问您想去哪里旅行呢？

【相关长期记忆】（用户画像事实与偏好记忆）
- [preference] 用户偏好海景住宿，且预算上限为每晚800元。 (final=0.0129, vector_sim=0.7718, rrf=0.0167)
- [fact] 用户计划下个月前往三亚旅游。 (final=0.0124, vector_sim=0.7540, rrf=0.0164)

【当前问题】
你还记得我喜欢住什么样的酒店吗？



In [11]:
RUN_LLM = True  # Set to True to have the LLM answer from the prompt

if RUN_LLM:
    print(extract_response_text(llm.invoke([{"role": "user", "content": prompt}])))
else:
    print("RUN_LLM=False：仅展示 Prompt。改 True 可体验完整闭环。")

记得的。根据我的记录，您偏好**海景住宿**，并且预算上限为**每晚800元**。

另外，我也注意到您计划**下个月前往三亚旅游**。结合您的喜好，三亚有很多符合“海景”且价格在800元以内的优质酒店或民宿选择。如果您需要，我可以为您筛选一些具体的住宿推荐。


## 小结

1. `memory_extract.py` 将非结构化对话抽取为结构化用户画像记忆。
2. `memory_merge.py` 合并相似记忆并保留独特信息。
3. `memory_store.py` 管理 Chroma 持久化、重复检测和索引缓存。
4. `memory_retrieve.py` 结合语义相似度与关键词匹配提高召回稳定性。
5. `memory_use.py` 将短期对话与长期画像记忆共同注入提示词。
6. 上述模块通过明确的输入输出边界衔接，使每个记忆阶段都可以独立理解和验证。
